In [0]:
#https://datahub.io/core/glacier-mass-balance

In [0]:
import requests
from pyspark.sql import DataFrame

In [0]:
with requests.get('https://datahub.io/core/glacier-mass-balance/_r/-/data/glaciers.csv',stream=True) as r: #reading as stream and not json
  with open('/tmp/glacier.csv', 'wb') as f:   #writing bytes of files in a file with specifed chunk size 8 MB data
    for chunk in r.iter_content(chunk_size=8192):
      f.write(chunk)

In [0]:
#same thing wrapped to a function
def get_data(url:str):
  filename = url.split('/')[-1]
  with requests.get('https://datahub.io/core/glacier-mass-balance/r/glaciers.csv', stream=True) as r:
    with open("/tmp/{}".format(filename), 'wb') as f:
      for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)
  return filename

In [0]:
file_name = get_data('https://datahub.io/core/glacier-mass-balance/r/glaciers.csv')

In [0]:
file_name

In [0]:
#now reading the file from the file system
spark.read.format("csv").option("header","true").load("file:/tmp/glacier.csv")

In [0]:
file_format = file_name.split(".")[-1]

In [0]:
#EXTRA: for just other file extensions
def read_data(file_name):
  if file_format == 'csv':
    df = spark.read.format(file_format).option("header","true").load("file:/tmp/{}".format(file_name))
  elif file_format == 'json':
    try:
      df = spark.read.format(file_format).load("file:/tmp{}".format(file_name))
    except:
      df = spark.read.format(file_format).option("multiline","true").load("file:/tmp/{}".format(file_name))
  elif file_format == 'parquet':
    df = spark.read.format(file_format).load("file:/tmp/{}".format(file_name))
  elif file_format == 'txt':
    df = spark.read.text("file:/dbfs/{}".format(file_name))
  return df

In [0]:
df = read_data(file_name)

In [0]:
# Download glacier CSV to /tmp
data_url = 'https://datahub.io/core/glacier-mass-balance/_r/-/data/glaciers.csv'
import requests
with requests.get(data_url, stream=True) as r:
    with open('/tmp/glacier.csv', 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# Try reading the CSV directly in Python (not Spark, since Spark cannot access /tmp or file: paths on AWS serverless)
import pandas as pd
df = pd.read_csv('/tmp/glacier.csv')
display(df.head(5))

In [0]:
# Fix: Use pandas API, not Spark
# Pandas DataFrame does not have createOrReplaceTempView. No change in logic — just workaround for error.
df.head()

In [0]:
# Fix: Register Spark temp view "df" for SQL cell
# Convert pandas DataFrame to Spark, register as 'df', then run SQL in next cell
from pyspark.sql import SparkSession
spark_df = spark.createDataFrame(df)
spark_df.createOrReplaceTempView("df")
print("Temp view 'df' created.")

In [0]:
%sql
select * from df

In [0]:

%sql
create or replace temp view nintys as select * from df where Year like '19%' order by Year asc;
create or replace temp view modern as select * from df where Year like '20%' order by Year asc;

In [0]:
nintys_df = spark.sql("select * from nintys")
modern_df = spark.sql("select * from modern")

In [0]:
def transform_data(df: DataFrame):
  spark.sql("create or replace temp view nintys as select * from df where Year like '19%' order by Year asc;")
  nintys_df = spark.sql("select * from nintys")
  spark.sql("create or replace temp view modern as select * from df where Year like '20%' order by Year asc;")
  modern_df = spark.sql("select * from modern")
  return nintys_df, modern_df

In [0]:
x, y = transform_data(df)

In [0]:
display(y)

In [0]:
display(x)

In [0]:
display(nintys_df)

In [0]:
display(modern_df)

In [0]:

nintys_file_namez = spark.sql("(select * from nintys order by Year ASC limit 1) union (select * from nintys order by Year DESC limit 1)")
modern_file_namez = spark.sql("(select * from modern order by Year ASC limit 1) union (select * from modern order by Year DESC limit 1)")

In [0]:
display(nintys_file_namez)

In [0]:
display(modern_file_namez)

In [0]:
modern_file_namez_df = modern_file_namez.collect()

In [0]:
str(modern_file_namez_df[0].__getitem__('Year')) + "-" + str(modern_file_namez_df[1].__getitem__('Year'))

In [0]:
def create_file_names():
  nintys_file_namez = spark.sql("(select * from nintys order by Year ASC limit 1) union (select * from nintys order by Year DESC limit 1)")
  modern_file_namez = spark.sql("(select * from modern order by Year ASC limit 1) union (select * from modern order by Year DESC limit 1)")
  nintys_file_namez_df = nintys_file_namez.collect()
  modern_file_namez_df = modern_file_namez.collect()
  nintys_file_name = nintys_file_namez_df[0].__getitem__('Year') + "-" + nintys_file_namez_df[1].__getitem__('Year') 
  modern_file_name = modern_file_namez_df[0].__getitem__('Year') + "-" + modern_file_namez_df[1].__getitem__('Year') 
  return nintys_file_name, modern_file_name

In [0]:
def create_file_names():
  nintys_file_namez = spark.sql("(select * from nintys order by Year ASC limit 1) union (select * from nintys order by Year DESC limit 1)")
  modern_file_namez = spark.sql("(select * from modern order by Year ASC limit 1) union (select * from modern order by Year DESC limit 1)")
  nintys_file_namez_df = nintys_file_namez.collect()
  modern_file_namez_df = modern_file_namez.collect()
  nintys_file_name = str(nintys_file_namez_df[0].__getitem__('Year')) + "-" + str(nintys_file_namez_df[1].__getitem__('Year'))
  modern_file_name = str(modern_file_namez_df[0].__getitem__('Year')) + "-" + str(modern_file_namez_df[1].__getitem__('Year'))
  return nintys_file_name, modern_file_name

In [0]:
modern_file_namez_df = modern_file_namez.collect()
nintys_file_namez_df = nintys_file_namez.collect()
m = str(modern_file_namez_df[0].__getitem__('Year')) + '-' + str(modern_file_namez_df[1].__getitem__('Year'))
n = str(nintys_file_namez_df[0].__getitem__('Year')) + '-' + str(nintys_file_namez_df[1].__getitem__('Year'))
print(m, n)

In [0]:
nintys_df.write.format('parquet').save("/dbfs/nintys_df.parquet")
def write_df(file_type: str,dfs, file_names):
  for x,y in zip(dfs,file_names):
    m = x.write.format(file_type).save("/tmp/{}.{}".format(y, file_type))
  return m 

write_df("parquet",[x, y],[m, n])

In [0]:
dbutils.fs.ls('/tmp/')